## Part 2: Model Interpretation with SHAP and LIME

In this section, we interpret the predictions of our fine-tuned BERT model by analyzing the word importances for various voxels using SHAP and LIME. Our goal is to gain insight into how the model makes its predictions and which words are most influential for predicting the fMRI responses in specific regions of the brain. While Part 1 focused on comparing overall predictive performance across different models and embedding methods, Part 2 takes a deeper look at model interpretability, aligning with the "P" (Predictability) principle of the PCS framework. We restrict our interpretability analysis to well-performing voxels to ensure that our explanations are based on reliable model outputs.

Among all the models evaluated in Part 1, we selected the **fine-tuned BERT model for Subject 3** for interpretation. This choice was based on its consistently superior performance across all evaluation metrics:
- Mean correlation coefficient (CC): 0.4301  
- Median CC: 0.4178  
- Top 5% CC: 0.5162  
- Top 1% CC: 0.5594  

These results were the best across all models, including the LoRA-based models and the Subject 2 versions, indicating that the fine-tuned BERT for Subject 3 was the most effective at capturing voxel-level brain responses. Therefore, we use this model as the basis for our interpretation experiments.

We will first select a test story and identify the top-performing voxels for that story based on the correlation coefficients. Then, we will use SHAP and LIME to investigate which words most strongly influence the model's predictions for those voxels. Finally, we will repeat the analysis for a second test story and compare the interpretation results across voxels and methods.

### Step 1: Select Well-Performing Voxels for a Given Test Story

To begin our interpretation, we first select a test story from the dataset—for example, `"fables_01"`. For this story, we aim to identify the voxels where the model performs well, meaning those for which the predicted fMRI responses show a high correlation with the true responses.

To do this, we use the voxel-level correlation coefficients (CC values) computed in Part 1 using the fine-tuned BERT model for Subject 3. The `voxel_cc.npy` file provides these CC values across all voxels. We sort the voxels by their CC scores and select the top-performing ones (e.g., the top 5 voxels) for further SHAP and LIME analysis.

By focusing on high-performing voxels, we ensure that our interpretation is grounded in predictions the model is making with reasonable accuracy—consistent with the "Performance" principle in the PCS framework.

In [14]:
import numpy as np
import joblib
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split
import sys
import importlib.util
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import BertTokenizer, BertModel

# ------------------------------------------------------------
# 0.  Configuration
# ------------------------------------------------------------
target_stories   = ["birthofanation",
                    "learninghumanityfromdogs",
                    "thesecrettomarriage"]

subject          = "subject3"
subject_dir      = Path("/jet/home/jlee45/tmp_ondemand_ocean_mth240012p_symlink/shared/data/subject3")
bert_ckpt_path   = Path("../data/bert_finetuned.pth")
output_dir       = Path("part2_voxel_selection")
output_dir.mkdir(parents=True, exist_ok=True)

alpha_grid       = np.logspace(0, 3, 20)  # same grid as before
n_boots          = 10
chunk_len        = 10
n_chunks         = 2
top_k            = 5                      # voxels to keep per story
device           = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# 1.  Define custom model and load fine-tuned weights
# ------------------------------------------------------------

class BERTVoxelRegressor(nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.linear = nn.Linear(self.bert.config.hidden_size, output_dim)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        return self.linear(cls)

# Set output_dim (number of voxels for subject3)
sample_story = target_stories[0]
sample_y = np.load(subject_dir / f"{sample_story}.npy")
output_dim = sample_y.shape[1]

# Initialize and load model
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
model = BERTVoxelRegressor(output_dim=output_dim)
model.load_state_dict(torch.load(bert_ckpt_path, map_location="cpu"))
model.to(device).eval()


# ------------------------------------------------------------
# 2.  Helper functions
# ------------------------------------------------------------
def word_level_embeddings(words):
    """Return an (n_words, 768) matrix of [CLS] embeddings."""
    all_vecs = []
    for w in words:
        toks = tokenizer(w, return_tensors="pt").to(device)
        with torch.no_grad():
            cls = model.bert(**toks).last_hidden_state[:, 0, :].cpu().numpy()
        all_vecs.append(cls.squeeze(0))
    return np.vstack(all_vecs)

def downsample_features(X_words, word2tr):
    """
    Average word-level embeddings within each TR indicated
    by 'word2tr' (length n_words → TR index per word).
    """
    n_trs = max(word2tr) + 1
    d     = X_words.shape[1]
    X_ds  = np.zeros((n_trs, d))
    counts = np.zeros(n_trs)
    for w_idx, tr_idx in enumerate(word2tr):
        X_ds[tr_idx] += X_words[w_idx]
        counts[tr_idx] += 1
    counts[counts == 0] = 1
    X_ds /= counts[:, None]
    return X_ds

def make_delayed(X, delays=range(1, 5)):
    """Concatenate delayed copies of X along the feature axis."""
    T, d = X.shape
    out  = [X]
    for k in delays:
        pad = np.zeros((k, d))
        out.append(np.vstack([pad, X[:-k]]))
    return np.hstack(out)

def zscore(M):
    s = M.std(axis=0); s[s == 0] = 1
    return (M - M.mean(axis=0)) / s

def load_story_fmri(story_name):
    return np.load(subject_dir / f"{story_name}.npy")

# ------------------------------------------------------------
# 3.  Load raw text & word-to-TR mapping
# ------------------------------------------------------------
with open("../data/raw_text.pkl", "rb") as f:
    raw_text = pickle.load(f)            # story_id → list[str]

with open("../data/word2tr.pkl", "rb") as f:
    word2tr = pickle.load(f)            # story_id → list[int]

# ------------------------------------------------------------
# 4.  Ridge utilities
# ------------------------------------------------------------
ridge_utils_dir = Path("ridge_utils")
sys.path.append(str(ridge_utils_dir))
spec = importlib.util.spec_from_file_location("ridge", ridge_utils_dir / "ridge.py")
ridge = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ridge)
bootstrap_ridge = ridge.bootstrap_ridge

# ------------------------------------------------------------
# 5.  Main loop over stories
# ------------------------------------------------------------
for story in target_stories:
    if story not in raw_text or story not in word2tr:
        print(f"[WARN]  {story} not found in raw_text or word2tr – skipping.")
        continue

    # 5-a.  Build delay-embedded feature matrix X
    words      = raw_text[story]
    X_words    = word_level_embeddings(words)                 # (n_words, 768)
    X_ds       = downsample_features(X_words, word2tr[story]) # (T, 768)
    X_trim     = X_ds[5:-10]                                  # drop first 5 s & last 10 s
    X_delay    = make_delayed(X_trim)                         # (T, 768*5)
    X_z        = zscore(X_delay)

    # 5-b.  Load and align fMRI data Y
    Y_full     = load_story_fmri(story)                       # original (T_full, V)
    Y_trim     = Y_full[5:-10]                                # match trim
    if X_trim.shape[0] != Y_trim.shape[0]:
        print(f"[WARN]  {story}: X and Y length mismatch – skipping.")
        continue
    Y_z        = zscore(Y_trim)

    # 5-c.  Run bootstrap_ridge
    _, corrs, _, _, _ = bootstrap_ridge(
        X_z, Y_z, X_z, Y_z,
        alphas=alpha_grid,
        nboots=n_boots,
        chunklen=chunk_len,
        nchunks=n_chunks,
        return_wt=False
    )  # corrs → (V,)

    # 5-d.  Select top-performing voxels
    corrs      = np.asarray(corrs)
    top_idx    = np.argsort(corrs)[-top_k:][::-1]

    # 5-e.  Save results
    np.save(output_dir / f"{story}_voxel_cc.npy", corrs)
    np.save(output_dir / f"{story}_top_voxel_idx.npy", top_idx)

    # 5-f.  Print summary
    print(f"\n{story}")
    for i in top_idx:
        print(f"  voxel {i:4d}  CC = {corrs[i]:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw_text.pkl'